In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [2]:
spark = SparkSession.builder.master("local[*]").appName("customerAnalysis").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/12 23:12:24 WARN Utils: Your hostname, Vishvas-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.52 instead (on interface en0)
26/08/12 23:12:24 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/12 23:12:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
spark

### Path:
```txt
/Users/vishvadubey/data/Other/Practice_DataSet/archive/olist_customers_dataset.csv
/Users/vishvadubey/data/Other/Practice_DataSet/archive/olist_geolocation_dataset
/Users/vishvadubey/data/Other/Practice_DataSet/archive/olist_order_items_dataset.csv
/Users/vishvadubey/data/Other/Practice_DataSet/archive/olist_order_payments_dataset.csv
/Users/vishvadubey/data/Other/Practice_DataSet/archive/olist_order_reviews_dataset.csv
/Users/vishvadubey/data/Other/Practice_DataSet/archive/olist_orders_dataset.csv
/Users/vishvadubey/data/Other/Practice_DataSet/archive/olist_products_dataset.csv
/Users/vishvadubey/data/Other/Practice_DataSet/archive/olist_sellers_dataset.csv
/Users/vishvadubey/data/Other/Practice_DataSet/archive/product_category_name_translation.csv
```
### file name:
```txt
olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_orders_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv
```

### Creating Dataframe using csv file - reading file

In [4]:
customers_dataset = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("/Users/vishvadubey/data/Other/Practice_DataSet/archive/olist_customers_dataset.csv")

geolocation_dataset = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("/Users/vishvadubey/data/Other/Practice_DataSet/archive/olist_geolocation_dataset.csv")

order_items_dataset = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("/Users/vishvadubey/data/Other/Practice_DataSet/archive/olist_order_items_dataset.csv")

order_payments_dataset = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("/Users/vishvadubey/data/Other/Practice_DataSet/archive/olist_order_payments_dataset.csv")

order_reviews_dataset = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("/Users/vishvadubey/data/Other/Practice_DataSet/archive/olist_order_reviews_dataset.csv")

orders_dataset = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("/Users/vishvadubey/data/Other/Practice_DataSet/archive/olist_orders_dataset.csv")

products_dataset = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("/Users/vishvadubey/data/Other/Practice_DataSet/archive/olist_products_dataset.csv")

sellers_dataset = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("/Users/vishvadubey/data/Other/Practice_DataSet/archive/olist_sellers_dataset.csv")

category_name_translation = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("/Users/vishvadubey/data/Other/Practice_DataSet/archive/product_category_name_translation.csv")

### creating the temp view using the Dataframe

In [5]:
customers_dataset.createOrReplaceTempView("customers_tbl")
geolocation_dataset.createOrReplaceTempView("geolocation_tbl")
order_items_dataset.createOrReplaceTempView("order_items_tbl")
order_payments_dataset.createOrReplaceTempView("order_payments_tbl")
order_reviews_dataset.createOrReplaceTempView("order_reviews_tbl")
orders_dataset.createOrReplaceTempView("orders_tbl")
products_dataset.createOrReplaceTempView("products_tbl")
sellers_dataset.createOrReplaceTempView("sellers_tbl")
category_name_translation.createOrReplaceTempView("category_name_translation_tbl")

In [6]:
# The number of unique repeat customers
customers_dataset.groupBy("customer_unique_id").count().filter(col("count") > 1).orderBy(col("count").desc()).show()

+--------------------+-----+
|  customer_unique_id|count|
+--------------------+-----+
|8d50f5eadf50201cc...|   17|
|3e43e6105506432c9...|    9|
|ca77025e7201e3b30...|    7|
|1b6c7548a2a1f9037...|    7|
|6469f99c1f9dfae77...|    7|
|63cfc61cee11cbe30...|    6|
|12f5d6e1cbf93dafd...|    6|
|de34b16117594161a...|    6|
|47c1a3033b8b77b3a...|    6|
|dc813062e0fc23409...|    6|
|f0e310a6839dce9de...|    6|
|fe81bb32c243a86b2...|    5|
|4e65032f1f574189f...|    5|
|b4e4f24de1e8725b7...|    5|
|394ac4de8f3acb142...|    5|
|56c8638e7c058b98a...|    5|
|74cb1ad7e6d567432...|    5|
|5e8f38a9a1c023f3d...|    5|
|35ecdf6858edc6427...|    5|
|b896655e2083a1d76...|    4|
+--------------------+-----+
only showing top 20 rows


In [7]:
customers_dataset\
    .join(orders_dataset, 
          customers_dataset["customer_id"]==orders_dataset["customer_id"], 
          how="inner")\
    .join(order_payments_dataset, 
          order_payments_dataset["order_id"]==orders_dataset["order_id"], 
          "inner")\
    .groupBy(col("customer_unique_id"))\
      .agg(avg(col("payment_value")))\
      .alias("average_payment")\
      .show()

+--------------------+------------------+
|  customer_unique_id|avg(payment_value)|
+--------------------+------------------+
|969cdc8af5b070747...|            134.66|
|f2a9bc9a1db05c873...|            122.42|
|d0c5e56e04e886e73...|             79.24|
|5f03b965e26e79a37...|              54.0|
|14a188558af6cd5bc...|122.57499999999999|
|ef1c2fafea5285a4b...|            235.84|
|d339ed835c9d8fd6b...|             262.1|
|39d6e50625a51a618...|             66.74|
|8d2fa65d968da66af...|            150.91|
|d04921557f1cde496...|            155.98|
|8033f408bcb751340...|              65.0|
|9ea65227cfac84dd2...|            119.79|
|2a6ef69674d6f2800...|             32.69|
|308dfe5168e217902...|            117.79|
|1d2435aa3b858d45c...|           9.22375|
|538f20c5da86daf31...|             72.84|
|968436577c1e2073b...|             91.26|
|1843cceebcc1e3912...|             99.43|
|86669cc06c6e824b3...|             61.36|
|c114c44566c2d172b...|           1234.62|
+--------------------+------------

26/08/13 00:48:26 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 928821 ms exceeds timeout 120000 ms
26/08/13 00:48:26 WARN SparkContext: Killing executors is not supported by current scheduler.
26/08/13 00:48:32 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$